# cusmic demo [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/cusmic/blob/main/demo/demo.ipynb)

Compare CuPy cusmic with L.A.Cosmic 1.4.0 and the committed FITS reference. In Colab, select a GPU runtime. Set `reference_mode="exact"` for v0.2.x or `"close"` for v0.3.x below; masks must match in both modes.

For local Jupyter, run from a repository checkout:

```sh
python -m pip install -e '.[cuda13,demo,bench]'
jupyter lab demo/demo.ipynb
```

Use `cuda12` for CUDA 12, or omit the CUDA extra if CuPy is installed. The full container includes this notebook, data, and tools under `/src`. Benchmark scripts and reference data come from the checkout or container, not the installed wheel.

To generate candidate references separately, install `.[test]` and run `make ref REFDIR=dist/reference`. See the [test guide](../test/README.md) for the reference policy and checks.


In [ ]:
import sys

tools_revision = "v0.2.6"  # Keep tools/data fixed across implementation versions.
revision = "v0.2.6"  # Installed implementation.
reference_mode = "exact"  # Use "close" for the v0.3.x contract.

if "google.colab" in sys.modules:
    %pip install -q "cupy-cuda12x==14.2.0" "cuda-toolkit[cudart,nvrtc,cccl]==12.6.3" astropy click matplotlib "lacosmic==1.4.0"
    %pip install -q --no-deps git+https://github.com/rndsrc/cusmic.git@{revision}


In [ ]:
from pathlib import Path
from urllib.request import urlopen

from cusmic.io import read_fits

root = Path.cwd()
if root.name == "demo":
    root = root.parent

base = f"https://raw.githubusercontent.com/rndsrc/cusmic/{tools_revision}"
for name in ("test/data/input.fits.gz", "test/data/error.fits.gz",
             "test/data/reference.fits.gz", "bench/bench.py",
             "bench/cpu.py"):
    path = root / name
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(urlopen(f"{base}/{name}").read())

sys.path.insert(0, str(root))
data = root / "test/data"
image, _ = read_fits(data / "input.fits.gz", dtype="float64")
error, _ = read_fits(data / "error.fits.gz", dtype="float64")
reference, _ = read_fits(data / "reference.fits.gz", dtype="float64")
reference_mask, _ = read_fits(data / "reference.fits.gz", ext="CRMASK")

In [ ]:
import cupy as cp
import lacosmic
import numpy as np
from astropy import log
from cusmic import __version__, remove_cosmics

log.setLevel("ERROR")
settings = dict(contrast=1, cr_threshold=5, neighbor_threshold=5, maxiter=4)

cpu_cleaned, cpu_mask = lacosmic.remove_cosmics(image, error=error, **settings)
cleaned, mask = remove_cosmics(cp.asarray(image), error=cp.asarray(error), **settings)
cleaned, mask = cp.asnumpy(cleaned), cp.asnumpy(mask)

np.testing.assert_array_equal(cpu_cleaned.view("uint64"), reference.view("uint64"))
np.testing.assert_array_equal(cpu_mask, reference_mask)
np.testing.assert_array_equal(mask, cpu_mask)
if reference_mode == "exact":
    np.testing.assert_array_equal(cleaned.view("uint64"), reference.view("uint64"))
elif reference_mode == "close":
    eps = 32 * np.finfo("float64").eps
    np.testing.assert_allclose(cleaned, reference, rtol=eps, atol=eps)
else:
    raise ValueError("reference_mode must be 'exact' or 'close'")

exact = np.array_equal(cleaned.view("uint64"), reference.view("uint64"))
print(f"cusmic {__version__}; CuPy {cp.__version__}")
print(f"Masks match; {mask.sum():,} detected pixels.")
print(f"Reference check ({reference_mode}) passed; bitwise equal: {exact}.")


In [ ]:
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize, PowerNorm

low, high = np.percentile(image, (2, 99.7))
intensity = PowerNorm(0.5, vmin=low, vmax=high)
removed = image - cleaned
signal = Normalize(0, max(1, removed.max()))
difference = cleaned - reference
scale = max(1e-12, np.abs(difference).max())

fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
panels = [
    (image, "Input", intensity, "gray"),
    (reference, "Cleaned L.A.Cosmic reference", intensity, "gray"),
    (cleaned, "Cleaned CuPy cusmic", intensity, "gray"),
    (image - reference, "Removed signal: reference", signal, "gray"),
    (removed, "Removed signal: CuPy", signal, "gray"),
    (difference, "Error: CuPy - reference", Normalize(-scale, scale), "RdBu_r"),
]
for ax, (pixels, title, norm, cmap) in zip(axes.flat, panels):
    shown = ax.imshow(pixels, origin="lower", norm=norm, cmap=cmap)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(shown, ax=ax, shrink=0.7)

## Benchmark

Both implementations run four warmups and sixteen measured calls on this frame. CPU timings use the installed L.A.Cosmic package. CuPy waits for GPU completion and measures uploads, resident cleaning, downloads, and complete calls. Progress and reference checks stay outside timing intervals.

The table reports median, mean, and sample standard deviation in milliseconds per frame. The speedup includes GPU transfers. The GPU is already initialized by the demo, so the first-result time is not a cold-start measurement.

For fresh-process reports with 1, 4, and 16 frames, run `CUSMIC_REFERENCE=exact python -m bench.run --backends cpu cupy` from the checkout (`close` for v0.3.x). `make bench REFERENCE=exact` also builds and measures CUDA C/C++. See the [benchmark guide](../bench/README.md) for A/B comparisons.


In [ ]:
from bench.bench import benchmark as benchmark_cupy
from bench.cpu import benchmark as benchmark_cpu

cpu = benchmark_cpu(image, error, (reference, reference_mask),
                    frames=1, warmups=4, repeats=16)
gpu, _ = benchmark_cupy(image, error, settings, warmups=4, repeats=16)

print("\nMilliseconds per frame (warmed):")
print(f"{'Method':<22} {'Median':>9} {'Mean':>9} {'SD':>9}")
for label, record, stage in (
    ("CPU complete", cpu, "total_ms"),
    ("CuPy resident", gpu, "clean_ms"),
    ("CuPy complete", gpu, "total_ms"),
):
    times = np.asarray(record["milliseconds"][stage]["samples"])
    print(f"{label:<22} {np.median(times):9.3f} "
          f"{times.mean():9.3f} {times.std(ddof=1):9.3f}")

speedup = (cpu["milliseconds"]["total_ms"]["median"] /
           gpu["milliseconds"]["total_ms"]["median"])
print(f"CuPy complete-call speedup over CPU: {speedup:.2f}x")
